# 🚀 VEYRA-INSTRUCT: Overnight GPU Master Studio (Kaggle & Colab)
### Train a full-scale, intelligent 125M AI Assistant while you sleep

---

### 😴 HOW TO TRAIN OVERNIGHT (Zero Disconnection & Safe Completion)
If you keep an interactive browser tab open, your computer may sleep or disconnect.

**To let Kaggle train safely in the background while you sleep:**
1. In the right panel under **Notebook options**:
   - Set **Accelerator** to **`GPU T4 x2`** or **`GPU P100`**.
   - Set **Internet** to **`On`**.
2. At the top right of your screen, click **`Save Version`** -> Choose **`Save & Run All (Commit)`**.
3. **You can now close your laptop, shut down your computer, and go to sleep!**
4. Kaggle runs the entire notebook autonomously in a cloud server for up to 12 hours.
5. In the morning, open this notebook: all training is done, and your final `veyra_instruct_125m_model.zip` is waiting in the **Output** tab ready for download!

---

### 🎯 WHAT THIS FULL-SCALE RUN BUILDS:
1. **Stage 1 (Pretraining - 8,000 steps)**: Ingests 25,000 multi-domain documents (*FineWeb-Edu*, *Cosmopedia*, *FineMath*, *The Stack*, *SlimPajama*) to build broad English grammar, vocabulary, coding, and mathematical foundations.
2. **Stage 2 (Supervised Fine-Tuning - 1,500 steps)**: Automatically fine-tunes on **All-Mix** (*OpenHermes 2.5* for poems/creative writing/dialogue, *CodeAlpaca* for programming, *GSM8K* for step-by-step math reasoning).
3. **Stage 3 (Lightweight Model Export)**: Packages pure weights, tokenizer, and config into a small ~165 MB zip for fast download and instant execution in `python -m veyra`.

In [ ]:
# Step 1: Verify GPU Accelerator
import torch
print(f"PyTorch Version : {torch.__version__}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected  : {device_name} ({vram_gb:.2f} GB VRAM)")
    print(f"   CUDA Version  : {torch.version.cuda}")
else:
    print("⚠️ No GPU detected!")
    print("   On Kaggle: In the right sidebar under Notebook options -> Accelerator -> Select 'GPU T4 x2' or 'GPU P100'.")
    print("   Then restart the session.")

In [ ]:
# Step 2: Fresh Environment Reset & Dependency Installation
import os, sys
from pathlib import Path

# Reset to a clean working directory
%cd /kaggle/working
!rm -rf Veyra Zonareth

# Clone latest repository from GitHub
!git clone https://github.com/shreyashmane-dev/Veyra.git
%cd /kaggle/working/Veyra

# Install in editable mode + streaming dependencies
!pip install -e . -q
!pip install datasets tqdm pyyaml huggingface_hub -q

# Run system diagnostic
!python -m veyra.cli system
print("\n✅ VEYRA codebase & environment successfully verified!")

### ⚡ Step 3: Launch Overnight Master Pipeline
This single automated command runs:
1. **BPE Tokenizer Training** (8,192 optimal vocabulary with fast inverted index).
2. **Pretraining** for **8,000 steps** on 25,000 diverse educational, math, code, and textbook documents.
3. **Rolling Checkpointing** (`--max-to-keep 2`) to ensure Kaggle disk space never exceeds limits.
4. **Automated SFT Instruction Tuning** (`--run-sft --sft-steps 1500`) on OpenHermes + CodeAlpaca + GSM8K.
5. **Automated Packaging** into `veyra_instruct_125m_model.zip`.

In [ ]:
# Step 3: Run Full-Scale Pretraining + Automated SFT Instruction Tuning
!python scripts/run_kaggle_pipeline.py \
    --model 125m \
    --steps 8000 \
    --batch-size 8 \
    --num-docs 25000 \
    --save-interval 500 \
    --max-to-keep 2 \
    --run-sft \
    --sft-steps 1500

### 🌅 Step 4: Morning Capability Verification Test
When you wake up, run this cell (or check its output if running via `Save & Run All`) to see your model write poems, solve math reasoning problems, code in Python, and explain concepts.

In [ ]:
# Step 4: Morning Verification Test (Poems, Math, Coding, Dialogue)
from pathlib import Path
try:
    from veyra.inference.engine import VeyraInferenceEngine
except ImportError:
    from zonareth.inference.engine import ZonarethInferenceEngine as VeyraInferenceEngine

ckpt_candidates = [
    Path("/kaggle/working/veyra_instruct_125m/release/model.pt"),
    Path("/kaggle/working/checkpoints/veyra_instruct_125m/latest.pt"),
    Path("/kaggle/working/veyra_125m/model.pt"),
    Path("/kaggle/working/zonareth_125m/model.pt"),
    Path("/kaggle/working/zonareth_125m_release/model.pt"),
]
ckpt_file = next((p for p in ckpt_candidates if p.exists()), None)
tok_path = Path("data/tokenizer")

if ckpt_file and tok_path.exists():
    print(f"✅ Loading trained model: {ckpt_file}")
    engine = VeyraInferenceEngine.from_checkpoint(str(ckpt_file), str(tok_path))
    
    test_suite = [
        ("🎭 Poetry Generation", "Write a short, beautiful poem about artificial intelligence gazing at the stars."),
        ("🧮 Math Reasoning", "What is 45 multiplied by 16? Explain the calculation step-by-step."),
        ("💻 Python Programming", "Write a clean Python function to check if a number is prime."),
        ("🧠 Technical Explanation", "Explain how an operating system manages virtual memory using page tables."),
    ]
    
    print("\n" + "=" * 70)
    print("   VEYRA-INSTRUCT: MORNING CAPABILITY VERIFICATION")
    print("=" * 70)
    for category, prompt in test_suite:
        print(f"\n{category}:")
        print(f"Prompt   : {prompt}")
        print("-" * 50)
        response = engine.generate(prompt, context=[])
        print(f"Response : {response}")
        print("-" * 50)
else:
    print("❌ Checkpoint not found. Verify that Step 3 completed successfully.")

### 📥 Step 5: Download Model Package
Click the download button or download the `.zip` from Kaggle's **Output** tab on the right sidebar.

In [ ]:
# Step 5: Download & Verification
import os
from pathlib import Path
from IPython.display import display, HTML, Javascript, FileLink

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
instruct_zips = list(working_dir.glob("*instruct*.zip"))
model_zips = list(working_dir.glob("*_model.zip"))
all_zips = list(working_dir.glob("*.zip"))

target_zip = instruct_zips[0] if instruct_zips else (model_zips[0] if model_zips else (all_zips[0] if all_zips else None))

if target_zip:
    size_mb = target_zip.stat().st_size / (1024 * 1024)
    print("=" * 70)
    print(f"✅ READY FOR DOWNLOAD: {target_zip.name} ({size_mb:.2f} MB)")
    print("=" * 70)
    
    for z in all_zips:
        z_mb = z.stat().st_size / (1024 * 1024)
        tag = "(⭐ RECOMMENDED INSTRUCT MODEL)" if "instruct" in z.name else "(Base Model Archive)"
        print(f"   📦 {z.name} -> {z_mb:.2f} MB {tag}")

    try:
        display(FileLink(str(target_zip.relative_to(Path.cwd())) if target_zip.is_relative_to(Path.cwd()) else str(target_zip.name)))
    except Exception:
        pass

    html_button = f"""
    <div style='margin-top: 15px;'>
        <a href='/files/{target_zip.name}' download='{target_zip.name}'
           style='background-color: #10b981; color: white; padding: 12px 24px; font-weight: bold; border-radius: 8px; text-decoration: none; display: inline-block; font-size: 16px;'>
           ⬇️ Click Here to Download {target_zip.name} ({size_mb:.1f} MB)
        </a>
    </div>
    <p style='color: gray; font-size: 13px; margin-top: 8px;'>
       Note: You can also download anytime from the <b>Output</b> tab on the right sidebar!
    </p>
    """
    display(HTML(html_button))
else:
    print("No .zip archives found yet. Please run Step 3 first!")

### ☁️ Step 6: (Optional) Auto-Push to Hugging Face Hub

In [ ]:
# Step 6: Optional Cloud Push to Hugging Face Hub
HF_TOKEN = ""  # e.g., 'hf_xxxxxxxxxxxxxxxxxxxxxxxx'
HF_REPO = ""   # e.g., 'your-username/veyra-instruct-125m'

if HF_TOKEN and HF_REPO:
    from huggingface_hub import HfApi
    release_candidates = list(Path("/kaggle/working").glob("*release"))
    target_folder = release_candidates[0] if release_candidates else Path("checkpoints/veyra_instruct_125m/release")
    if target_folder.exists():
        print(f"Uploading {target_folder} to https://huggingface.co/{HF_REPO}...")
        api = HfApi(token=HF_TOKEN)
        api.create_repo(repo_id=HF_REPO, exist_ok=True, private=True)
        api.upload_folder(folder_path=str(target_folder), repo_id=HF_REPO)
        print(f"✅ Successfully uploaded to: https://huggingface.co/{HF_REPO}")
    else:
        print("Release folder not found. Run Step 3 first.")
else:
    print("HF_TOKEN and HF_REPO not set. Skipping cloud push (Local zip ready in Step 5).")